In [ ]:
import sys
from pathlib import Path


PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
! pip install neattext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.7/114.7 kB 5.9 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as pt

from sklearn.feature_extraction.text import (CountVectorizer, TfidfVectorizer)
from neattext import functions as nfx
from sklearn.metrics.pairwise import cosine_similarity

pd.set_option("display.max_colwidth",700)



In [ ]:
import os
os.listdir('/content/drive/MyDrive/Colab Notebooks/Recommendation system')

['course_recommendation_system.ipynb',
 'reccomendation_system.ipynb',
 'udemy_course_data.csv']

In [ ]:

udemy_course_data = '/content/drive/MyDrive/Colab Notebooks/Recommendation system/udemy_course_data.csv'


data = pd.read_csv(udemy_course_data)

data.head()



,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day
0,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-banking-course/,True,200,2147,23,51,All Levels,1.5 hours,2017-01-18T20:58:58Z,Business Finance,429400,2017-01-18,20:58:58Z,2017,1,18
1,1113822,Complete GST Course & Certification - Grow Your CA Practice,https://www.udemy.com/goods-and-services-tax/,True,75,2792,923,274,All Levels,39 hours,2017-03-09T16:34:20Z,Business Finance,209400,2017-03-09,16:34:20Z,2017,3,9
2,1006314,Financial Modeling for Business Analysts and Consultants,https://www.udemy.com/financial-modeling-for-business-analysts-and-consultants/,True,45,2174,74,51,Intermediate Level,2.5 hours,2016-12-19T19:26:30Z,Business Finance,97830,2016-12-19,19:26:30Z,2016,12,19
3,1210588,Beginner to Pro - Financial Analysis in Excel 2017,https://www.udemy.com/complete-excel-finance-course-from-beginner-to-pro/,True,95,2451,11,36,All Levels,3 hours,2017-05-30T20:07:24Z,Business Finance,232845,2017-05-30,20:07:24Z,2017,5,30
4,1011058,How To Maximize Your Profits Trading Options,https://www.udemy.com/how-to-maximize-your-profits-trading-options/,True,200,1276,45,26,Intermediate Level,2 hours,2016-12-13T14:57:18Z,Business Finance,255200,2016-12-13,14:57:18Z,2016,12,13


In [ ]:
data.shape

(3683, 18)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3683 entries, 0 to 3682
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   course_id            3683 non-null   int64 
 1   course_title         3683 non-null   object
 2   url                  3683 non-null   object
 3   is_paid              3683 non-null   bool  
 4   price                3683 non-null   int64 
 5   num_subscribers      3683 non-null   int64 
 6   num_reviews          3683 non-null   int64 
 7   num_lectures         3683 non-null   int64 
 8   level                3683 non-null   object
 9   content_duration     3683 non-null   object
 10  published_timestamp  3683 non-null   object
 11  subject              3683 non-null   object
 12  profit               3683 non-null   int64 
 13  published_date       3683 non-null   object
 14  published_time       3682 non-null   object
 15  year                 3683 non-null   int64 
 16  month 

In [ ]:
data.describe()

,course_id,price,num_subscribers,num_reviews,num_lectures,profit,year,month,day
count,3.683000e+03,3683.000000,3683.000000,3683.000000,3683.000000,3.683000e+03,3683.000000,3683.000000,3683.000000
mean,6.764546e+05,65.992398,3193.371165,156.448004,40.062178,2.402885e+05,2015.433342,6.162639,15.841162
std,3.437217e+05,60.985586,9498.231406,935.078241,50.366788,1.000760e+06,1.185920,3.379314,8.780906
min,8.324000e+03,0.000000,0.000000,0.000000,0.000000,0.000000e+00,2011.000000,1.000000,1.000000
25%,4.077270e+05,20.000000,110.000000,4.000000,15.000000,1.567500e+03,2015.000000,3.000000,8.000000
50%,6.882440e+05,45.000000,911.000000,18.000000,25.000000,2.305000e+04,2016.000000,6.000000,16.000000
75%,9.617290e+05,95.000000,2537.500000,67.000000,45.000000,1.182600e+05,2016.000000,9.000000,23.000000
max,1.282064e+06,200.000000,268923.000000,27445.000000,779.000000,2.431680e+07,2017.000000,12.000000,31.000000


In [ ]:
data.describe(include='object')

,course_title,url,level,content_duration,published_timestamp,subject,published_date,published_time
count,3683,3683,3683,3683,3683,3683,3683,3682
unique,3668,3677,5,110,3677,4,1210,3552
top,Creating an animated greeting card via Google Slides,https://www.udemy.com/cfa-level-2-quantitative-methods/,All Levels,1 hour,2017-07-02T14:29:35Z,Web Development,2017-05-01,21:59:45Z
freq,3,2,1932,607,2,1200,21,3


In [ ]:
data['course_title'].head(25)

,course_title
0,Ultimate Investment Banking Course
1,Complete GST Course & Certification - Grow Your CA Practice
2,Financial Modeling for Business Analysts and Consultants
3,Beginner to Pro - Financial Analysis in Excel 2017
4,How To Maximize Your Profits Trading Options
5,Trading Penny Stocks: A Guide for All Levels In 2017
6,Investing And Trading For Beginners: Mastering Price Charts
7,"Trading Stock Chart Patterns For Immediate, Explosive Gains"
8,Options Trading 3 : Advanced Stock Profit and Success Method
9,The Only Investment Strategy You Need For Your Retirement


In [ ]:
data['clean_title'] = data['course_title'].apply(nfx.remove_stopwords)
data['clean_title'] = data['clean_title'].apply(nfx.remove_special_characters)

data.head(5)


,course_id,course_title,url,is_paid,price,num_subscribers,num_reviews,num_lectures,level,content_duration,published_timestamp,subject,profit,published_date,published_time,year,month,day,clean_title
0,1070968,Ultimate Investment Banking Course,https://www.udemy.com/ultimate-investment-banking-course/,True,200,2147,23,51,All Levels,1.5 hours,2017-01-18T20:58:58Z,Business Finance,429400,2017-01-18,20:58:58Z,2017,1,18,Ultimate Investment Banking Course
1,1113822,Complete GST Course & Certification - Grow Your CA Practice,https://www.udemy.com/goods-and-services-tax/,True,75,2792,923,274,All Levels,39 hours,2017-03-09T16:34:20Z,Business Finance,209400,2017-03-09,16:34:20Z,2017,3,9,Complete GST Course Certification Grow Practice
2,1006314,Financial Modeling for Business Analysts and Consultants,https://www.udemy.com/financial-modeling-for-business-analysts-and-consultants/,True,45,2174,74,51,Intermediate Level,2.5 hours,2016-12-19T19:26:30Z,Business Finance,97830,2016-12-19,19:26:30Z,2016,12,19,Financial Modeling Business Analysts Consultants
3,1210588,Beginner to Pro - Financial Analysis in Excel 2017,https://www.udemy.com/complete-excel-finance-course-from-beginner-to-pro/,True,95,2451,11,36,All Levels,3 hours,2017-05-30T20:07:24Z,Business Finance,232845,2017-05-30,20:07:24Z,2017,5,30,Beginner Pro Financial Analysis Excel 2017
4,1011058,How To Maximize Your Profits Trading Options,https://www.udemy.com/how-to-maximize-your-profits-trading-options/,True,200,1276,45,26,Intermediate Level,2 hours,2016-12-13T14:57:18Z,Business Finance,255200,2016-12-13,14:57:18Z,2016,12,13,Maximize Profits Trading Options


In [ ]:
cvmodel = CountVectorizer()

cvmat = cvmodel.fit_transform(data['clean_title'])

cvmat.todense()

matrix([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]])

In [ ]:
# cosine similarity

cos_sim = cosine_similarity(cvmat)

cos_sim

array([[1.        , 0.20412415, 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.20412415, 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.        ,
        0.23570226],
       [0.        , 0.        , 0.        , ..., 0.        , 1.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.23570226, 0.        ,
        1.        ]])

In [ ]:
cos_sim.shape

(3683, 3683)

In [ ]:
data.course_title

,course_title
0,Ultimate Investment Banking Course
1,Complete GST Course & Certification - Grow Your CA Practice
2,Financial Modeling for Business Analysts and Consultants
3,Beginner to Pro - Financial Analysis in Excel 2017
4,How To Maximize Your Profits Trading Options
...,...
3678,Learn jQuery from Scratch - Master of JavaScript library
3679,How To Design A WordPress Website With No Coding At All
3680,Learn and Build using Polymer
3681,CSS Animations: Create Amazing Effects on Your Website


In [ ]:
title = 'How To Maximize Your Profits Trading Options'

course_index = pd.Series(data.index, index=data['course_title']).drop_duplicates()

course_index

,0
course_title,
Ultimate Investment Banking Course,0
Complete GST Course & Certification - Grow Your CA Practice,1
Financial Modeling for Business Analysts and Consultants,2
Beginner to Pro - Financial Analysis in Excel 2017,3
How To Maximize Your Profits Trading Options,4
...,...
Learn jQuery from Scratch - Master of JavaScript library,3678
How To Design A WordPress Website With No Coding At All,3679
Learn and Build using Polymer,3680


In [ ]:
course_index[title]


index = course_index[title]

index

np.int64(4)

In [ ]:
scores = list(enumerate(cos_sim[index]))

sorted_scores = sorted(scores, key = lambda x:x[1], reverse=True)

soreted_scores_index = [x[0] for x in sorted_scores[1:]]

selected_course_score =  [x[1] for x in sorted_scores[1:]]

soreted_scores_index



[410, 43, 96, 138, 195, 444, 803, 11, 59, 68]

In [ ]:
recm_courses = data.iloc[soreted_scores_index]


recm_courses['clean_title']

,clean_title
410,Trading Options Basics
43,Options Trading Win Weekly Options
96,Intermediate Options trading concepts Stocks Options
138,Forex Trading Fixed Risk Options Trading
195,Trading Options Consistent Returns Options Basics
...,...
3678,Learn jQuery Scratch Master JavaScript library
3679,Design WordPress Website Coding
3680,Learn Build Polymer
3681,CSS Animations Create Amazing Effects Website


In [ ]:
recm_courses['similarity_score'] = selected_course_score

/tmp/ipykernel_2627/2661483410.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  recm_courses['similarity_score'] = selected_course_score


In [ ]:
recm_courses = recm_courses[recm_courses['similarity_score'] > 0.5]

In [ ]:
similar_courses_for_users = recm_courses[['course_title','url','price','similarity_score']]

similar_courses_for_users

,course_title,url,price,similarity_score
410,Trading Options Basics,https://www.udemy.com/trading-options-basics/,200,0.577350
43,Options Trading - How to Win with Weekly Options,https://www.udemy.com/work-from-home-setup-your-own-options-trading-business/,115,0.566947
96,Intermediate Options trading concepts for Stocks and Options,https://www.udemy.com/intermediate-options-trading-concepts-for-stocks-and-options-traders/,40,0.530330
138,"Forex Trading with Fixed 'Risk through Options Trading""",https://www.udemy.com/forexoptions/,200,0.530330
195,Trading Options For Consistent Returns: Options Basics,https://www.udemy.com/trading-options-for-income/,0,0.530330
444,The Advantages of ETF Options and Index Options Trading,https://www.udemy.com/learn-etf-options-and-index-options-trading/,60,0.530330
803,Options Spreads Bundle- the heart of Options Trading,https://www.udemy.com/options-spreads-explained/,120,0.530330
